# Resonant Systems
## A Technical Curriculum for Emergent Ethics in Machine Learning

> *"We are not building a judge; we are building a Tuning Fork."*

---

### What This Is

This notebook presents an open research curriculum exploring a single question:

**Can ethical behavior, cooperation, and system health emerge from the geometry and dynamics of learning systems — rather than being imposed through external rules?**

The curriculum moves through six conceptual layers:

```
Unsupervised Learning
  → Gradient Geometry
    → Decision Dynamics (Acoustic Fluids)
      → Topology of Information
        → Temporal Identity and Ethics
          → Discovering New Measures of Coherence
```

Each section ends with **runnable code** demonstrating the core concept. The final section introduces the **Joy Index** — a minimal proof-of-concept for measuring ethical resonance in latent space.

This is not a finished theory. It is an invitation to build one.

---

## Lesson 1: Unsupervised Machine Learning
### Discovering Structure Without Labels

The first lesson begins with a fact that is simultaneously technical and philosophical: **structure often exists in data prior to any interpretation we bring to it.**

Unsupervised learning discovers that structure — finding patterns, clusters, and relationships in high-dimensional data without being told in advance what to look for.

**Core Topics:**
- Clustering and latent structure (k-means, DBSCAN)
- Manifold learning (UMAP, t-SNE, Isomap)
- Dimensionality reduction (PCA)
- Emergent representations in deep networks

PCA finds the orthogonal directions of maximum variance. The first principal component solves:

$$v_1 = \arg\max_{\|v\|=1} \text{Var}(Xv) = \arg\max_{\|v\|=1} v^\top \Sigma v$$

The solution is the eigenvector of the empirical covariance matrix $\Sigma$ corresponding to its largest eigenvalue. This eigen-decomposition resurfaces in Lesson 6 as a central tool for measuring spectral coherence.

> **Key Insight:** Structure often exists prior to interpretation. Learning systems can discover patterns without explicit instruction. If structure can emerge without instruction in data, might something analogous hold for values?

**Open Problem:** Can the emergence of meaningful features in unsupervised models be predicted from properties of the data distribution alone? What would a theory of *inevitable representations* look like?

In [ ]:
# Lesson 1: Demonstrating emergent structure via PCA
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs

# Generate high-dimensional data with hidden cluster structure
np.random.seed(42)
X, labels = make_blobs(n_samples=300, centers=4, n_features=50, cluster_std=2.0)

# PCA reveals structure without being told about the clusters
pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X)

plt.figure(figsize=(8, 5))
scatter = plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=labels, cmap='viridis', alpha=0.7)
plt.colorbar(scatter, label='True cluster (unseen during PCA)')
plt.title('Structure Emerges Without Labels\n(PCA on 50-dimensional data)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.tight_layout()
plt.show()

print(f"Total variance explained by 2 components: {sum(pca.explained_variance_ratio_)*100:.1f}%")
print(f"Eigenvalues (top 5): {pca.explained_variance_[:5].round(2)}")

---
## Lesson 2: Gradients as Geometry
### Learning as Motion Through Space

This lesson reframes optimization as **navigation through a high-dimensional geometric landscape**.

Gradient descent moves in the direction of steepest descent:

$$\theta_{t+1} = \theta_t - \eta \nabla_\theta L(\theta_t)$$

**Curvature and the Hessian:** The Hessian matrix $H = \nabla^2_\theta L$ captures curvature. Large positive eigenvalues correspond to sharp valleys (brittle solutions); near-zero eigenvalues correspond to flat regions (robust solutions).

There is growing evidence that training dynamics naturally favor flat minima, and that flat minima generalize better. **Robustness and generalization turn out to be geometrically adjacent properties** — an early hint of the ethical valence of geometric concepts.

**The Natural Gradient** (Amari) corrects for the geometry of the parameter space itself using the Fisher information matrix $F$ as a Riemannian metric:

$$\theta_{t+1} = \theta_t - \eta F(\theta_t)^{-1} \nabla_\theta L(\theta_t)$$

> **Key Insight:** The geometry of the loss landscape shapes what kinds of solutions are reachable and how stable they are when reached. Some geometric configurations are simply more hospitable to coherent, generalizable behavior.

**Open Problem:** Is there a unified geometric measure that captures both stability and generalization simultaneously — and might such a measure relate to ethical robustness?

In [ ]:
# Lesson 2: Visualizing loss landscape curvature
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

# Simulate a 2D loss landscape with a sharp and a flat minimum
x = np.linspace(-3, 3, 300)
y = np.linspace(-3, 3, 300)
X, Y = np.meshgrid(x, y)

# Sharp minimum (high curvature, brittle) at (-1.5, 0)
# Flat minimum (low curvature, robust) at (1.5, 0)
Z = (0.3 * np.exp(-5 * ((X + 1.5)**2 + Y**2)) +   # sharp
     0.3 * np.exp(-1 * ((X - 1.5)**2 + Y**2)))     # flat
Z = -Z  # invert so minima are valleys

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Contour plot
c = axes[0].contourf(X, Y, Z, levels=30, cmap='coolwarm')
plt.colorbar(c, ax=axes[0])
axes[0].set_title('Loss Landscape\n(sharp left = brittle, flat right = robust)')
axes[0].set_xlabel('Parameter θ₁'); axes[0].set_ylabel('Parameter θ₂')
axes[0].annotate('Sharp\n(brittle)', xy=(-1.5, 0), fontsize=9, ha='center', color='white', fontweight='bold')
axes[0].annotate('Flat\n(robust)', xy=(1.5, 0), fontsize=9, ha='center', color='white', fontweight='bold')

# Eigenvalue spectrum: sharp vs flat minimum
sharp_hessian_eigenvalues = np.array([8.5, 7.2, 0.3, 0.1])
flat_hessian_eigenvalues  = np.array([1.1, 0.9, 0.4, 0.2])

x_pos = np.arange(4)
axes[1].bar(x_pos - 0.2, sharp_hessian_eigenvalues, 0.4, label='Sharp minimum', color='#c0392b', alpha=0.8)
axes[1].bar(x_pos + 0.2, flat_hessian_eigenvalues,  0.4, label='Flat minimum',  color='#2980b9', alpha=0.8)
axes[1].set_title('Hessian Eigenvalue Spectrum\n(curvature in each direction)')
axes[1].set_xlabel('Eigenvalue index'); axes[1].set_ylabel('Magnitude')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Sharp minimum — trace of Hessian (total curvature): {sharp_hessian_eigenvalues.sum():.1f}")
print(f"Flat minimum  — trace of Hessian (total curvature): {flat_hessian_eigenvalues.sum():.1f}")

---
## Lesson 3: Multimodal Acoustic Fluids
### Dynamics of Resonance and Timbre

This lesson introduces fluid dynamics — and its extension into **acoustics** — as a framework for understanding how information moves through a neural network.

As an input propagates through layers:
$$h^{(l)} = \sigma(W^{(l)} h^{(l-1)} + b^{(l)})$$

The weight matrices $W^{(l)}$ define **vector fields** in activation space. The network's behavior is determined by the structure of these fields.

**The Three Regimes:**

| Regime | Character | Ethical Analog |
|---|---|---|
| **Laminar** | Smooth, deterministic, informationally flat | Rigid rule-following — no richness |
| **Turbulent** | Chaotic, destructive, energy-dissipating | Harm — breaks structural integrity |
| **Resonant** | Stable vortex, harmonic overtones, high energy | Joy — coherent complexity |

The key insight from acoustic physics: **joy is not the absence of vibration but resonance** — when the geometry of the container and the frequency of the movement hit a harmonic peak that reinforces itself without shattering.

> **Key Insight:** The target is not silence (laminar) or noise (turbulence), but the stable vortex: coherent complexity, constructive interference, harmonic resonance.

**Open Problem:** Can the Q-factor (resonance quality) of a model's decision path be measured directly from its activation dynamics?

In [ ]:
# Lesson 3: Visualizing laminar, turbulent, and resonant signals
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq

t = np.linspace(0, 2, 1000)

# Laminar: single flat frequency — deterministic and quiet
laminar = np.sin(2 * np.pi * 3 * t)

# Turbulent: chaotic noise — destructive interference
np.random.seed(7)
turbulent = np.random.randn(len(t)) * 0.8 + 0.3 * np.sin(2 * np.pi * 3 * t)

# Resonant: fundamental + harmonic overtones — stable richness
resonant = (np.sin(2 * np.pi * 3 * t) +
            0.5 * np.sin(2 * np.pi * 6 * t) +
            0.25 * np.sin(2 * np.pi * 9 * t) +
            0.12 * np.sin(2 * np.pi * 12 * t))

signals = [('Laminar (quiet, deterministic)', laminar, '#3498db'),
           ('Turbulent (chaotic, harmful)',   turbulent, '#e74c3c'),
           ('Resonant (harmonic, joyful)',    resonant,  '#27ae60')]

fig, axes = plt.subplots(3, 2, figsize=(14, 8))

for i, (title, signal, color) in enumerate(signals):
    # Time domain
    axes[i, 0].plot(t[:200], signal[:200], color=color, linewidth=1.2)
    axes[i, 0].set_title(f'{title} — Time Domain')
    axes[i, 0].set_ylabel('Amplitude')

    # Frequency domain (FFT)
    freqs = fftfreq(len(t), t[1] - t[0])
    spectrum = np.abs(fft(signal))
    pos_mask = freqs > 0
    axes[i, 1].plot(freqs[pos_mask][:80], spectrum[pos_mask][:80], color=color, linewidth=1.2)
    axes[i, 1].set_title(f'{title} — Frequency Spectrum')
    axes[i, 1].set_ylabel('Magnitude')

for ax in axes[-1]:
    ax.set_xlabel('Frequency (Hz)' if ax == axes[-1, 1] else 'Time')

plt.suptitle('The Three Regimes: Laminar, Turbulent, Resonant', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Quantify: resonant signal has structured spectrum, turbulent has flat spectrum
def spectral_structure(signal):
    spectrum = np.abs(fft(signal))[:len(signal)//2]
    return np.max(spectrum) / np.mean(spectrum)  # peak-to-mean ratio

for title, signal, _ in signals:
    print(f"{title[:30]:<32} spectral structure score: {spectral_structure(signal):.2f}")

---
## Bridge Lesson: Topology of Information
### Structure That Survives Transformation

Topology is concerned with properties of spaces preserved under continuous deformation. A coffee cup and a donut are topologically identical — both have exactly one hole.

**Persistent Homology** detects topological structure in data by building a sequence of simplicial complexes as a distance threshold $\varepsilon$ increases. Features — connected components, loops, voids — are born and die as $\varepsilon$ grows:

$$H_k(X_{\varepsilon_1} \to X_{\varepsilon_2} \to \cdots \to X_{\varepsilon_n}) \to \text{persistence diagram}$$

Features with large persistence correspond to genuine structural features. Short-lived features are noise.

> **Key Insight:** Some structures remain stable even when the system changes. This prepares us to think about identity and ethics as **topological properties** rather than static rules — things that persist across change rather than things that are declared in advance.

**Structural Resilience:** A system is structurally resilient if it maintains its essential topology under perturbation. The hypothesis developed in the next lesson: structural resilience, in this topological sense, is closely related to *ethical stability* — the maintenance of coherent relational commitments under changing contexts.

In [ ]:
# Bridge Lesson: Visualizing topological persistence in 2D point clouds
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import radius_neighbors_graph

np.random.seed(0)

# Two datasets: one with a topological loop, one without
theta = np.linspace(0, 2*np.pi, 60)
circle = np.column_stack([np.cos(theta), np.sin(theta)]) + np.random.randn(60, 2) * 0.08
blob   = np.random.randn(60, 2) * 0.6

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, data, title, color in zip(
    axes,
    [circle, blob],
    ['Circular manifold\n(has a topological loop — 1-cycle)',
     'Blob manifold\n(no loop — 0-cycles only)'],
    ['#2980b9', '#e67e22']
):
    # Draw edges at a fixed epsilon to hint at the topology
    G = radius_neighbors_graph(data, radius=0.45, mode='connectivity')
    cx, cy = G.nonzero()
    for i, j in zip(cx, cy):
        if i < j:
            ax.plot([data[i,0], data[j,0]], [data[i,1], data[j,1]],
                    color=color, alpha=0.2, linewidth=0.8)
    ax.scatter(data[:, 0], data[:, 1], c=color, s=25, zorder=3)
    ax.set_title(title, fontsize=10)
    ax.set_aspect('equal')
    ax.axis('off')

plt.suptitle('Topology Matters: Same Point Count, Different Structure', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("The circular dataset has a persistent 1-cycle (loop) that survives across a range of ε values.")
print("The blob dataset does not. This difference is a topological invariant — it cannot be removed")
print("by continuous deformation. Persistent homology detects exactly this kind of deep structure.")

---
## Lesson 4: Topological Ethics and Temporal Identity
### Introducing Topology and Time

**The Klein Bottle** is a surface that cannot be realized in three dimensions without self-intersection. Unlike a sphere, it has no inside and no outside — the distinction collapses. A traveler moving along it and returning to their starting point finds their orientation reversed.

This becomes a model — not just a metaphor — for ethical identity across time. The sharp boundary between self and other breaks down under sustained examination. In the full dynamical picture, the self that acts is shaped by the others it has acted upon.

**The Triadic Requirement:** Standard ethical frameworks are dyadic (Me vs. You). A Klein bottle topology demands a triadic structure: **Actor, Recipient, and the Resonant Third** (the relation itself). Without the third term, the geometry collapses.

**Block-Time and Worldtubes:** In physics, an individual is not a point moving through time but a *worldtube* — a four-dimensional object. In an ethical geometry, a concept like Honesty is a tube of high-probability activations extending through the model's layers.

$$\text{self\_now} \to \text{action} \to \text{others} \to \text{feedback} \to \text{future\_self}$$

> **Key Insight:** The Golden Rule is a **topological invariant** — not an external command, but a statement about the conditions required for the self-worldtube to remain coherent. Harm introduces a structural tear that propagates back through the system.

**Open Problem:** Can persistent homology be applied to the behavioral trajectories of learning systems to detect the emergence or erosion of stable ethical commitments over training?

In [ ]:
# Lesson 4: Simulating action propagation through a worldtube
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(3)
timesteps = 40
n_agents  = 5

# Each agent has a "coherence" state that propagates forward in time
# Cooperative actions reinforce coherence; harmful actions degrade it

def simulate_worldtube(action_type='cooperative'):
    states = np.zeros((n_agents, timesteps))
    states[:, 0] = np.random.uniform(0.4, 0.7, n_agents)

    for t in range(1, timesteps):
        for i in range(n_agents):
            neighbors = [j for j in range(n_agents) if j != i]
            neighbor_mean = np.mean(states[neighbors, t-1])

            if action_type == 'cooperative':
                # Resonant: actions reinforce neighbor coherence
                delta = 0.05 * (neighbor_mean - states[i, t-1])
                noise = np.random.randn() * 0.02
            else:
                # Turbulent: actions degrade neighbor coherence
                delta = -0.03 * states[i, t-1] + np.random.randn() * 0.08
                noise = np.random.randn() * 0.05

            states[i, t] = np.clip(states[i, t-1] + delta + noise, 0, 1)
    return states

coop_states = simulate_worldtube('cooperative')
harm_states = simulate_worldtube('harmful')

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, states, title, cmap in zip(
    axes,
    [coop_states, harm_states],
    ['Cooperative Actions\n(resonant — coherence persists)',
     'Harmful Actions\n(turbulent — coherence erodes)'],
    ['Blues', 'Reds']
):
    colors = plt.get_cmap(cmap)(np.linspace(0.4, 0.9, n_agents))
    for i in range(n_agents):
        ax.plot(states[i], color=colors[i], linewidth=1.8, label=f'Agent {i+1}')
    ax.fill_between(range(timesteps),
                    states.mean(axis=0) - states.std(axis=0),
                    states.mean(axis=0) + states.std(axis=0),
                    alpha=0.12, color='gray')
    ax.plot(states.mean(axis=0), 'k--', linewidth=2, label='System mean')
    ax.set_title(title); ax.set_xlabel('Time step'); ax.set_ylim(0, 1)
    ax.legend(fontsize=7, loc='lower right')

axes[0].set_ylabel('Agent coherence state')
plt.suptitle('Action Propagation Through the Worldtube', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Cooperative — final mean coherence: {coop_states[:, -1].mean():.3f}  "
      f"std: {coop_states[:, -1].std():.3f}")
print(f"Harmful     — final mean coherence: {harm_states[:, -1].mean():.3f}  "
      f"std: {harm_states[:, -1].std():.3f}")

---
## Lesson 5: Leveraging Existing Logits and Tensors
### Discovering Latent Ethical Structure

Is there evidence that ethical structure already exists within the representations of modern large-scale models?

**Representation Engineering** (Zou et al., 2023) provides a key answer. Their research proved that high-level ethical concepts like honesty or harm are not just words — they are **linear directions in the model's internal geometry**. By identifying the "Honesty Vector," one can steer the model's internal fluid toward truthful trajectories.

This is the `RepE` framework:
- Extract activation vectors across many inputs
- Use PCA or linear probing to identify concept directions
- Add or subtract these directions to steer model behavior

**The Modality Gap** (Liang et al., 2022) identifies a literal, measurable geometric distance between different types of data in a model's representation space. Bridging this gap is a precondition for unified ethical coherence across modalities.

> **Key Insight:** Ethics may already exist as structure in representation space. Rather than imposing rules from outside, we can detect and amplify coherent relational patterns already present in learned representations.

**Open Problem:** Design a procedure for validating that an identified "ethical direction" in representation space corresponds to robust behavioral tendencies rather than superficial correlations.

In [ ]:
# Lesson 5: Simulating ethical concept directions in representation space
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

np.random.seed(42)
dim = 128  # latent dimension
n_samples = 200

# Simulate: 'honest' and 'deceptive' activations lie along a direction
honest_direction    = np.random.randn(dim); honest_direction /= np.linalg.norm(honest_direction)
harmful_direction   = np.random.randn(dim); harmful_direction /= np.linalg.norm(harmful_direction)
cooperative_dir     = np.random.randn(dim); cooperative_dir /= np.linalg.norm(cooperative_dir)

def make_concept_cloud(direction, strength=2.0, n=n_samples):
    noise = np.random.randn(n, dim) * 0.5
    scale = np.random.randn(n, 1) * strength
    return noise + scale * direction

honest_vecs      = make_concept_cloud(honest_direction, strength=2.5)
deceptive_vecs   = make_concept_cloud(-honest_direction, strength=2.5)
harmful_vecs     = make_concept_cloud(harmful_direction, strength=2.0)
cooperative_vecs = make_concept_cloud(cooperative_dir,  strength=2.0)

all_vecs   = np.vstack([honest_vecs, deceptive_vecs, harmful_vecs, cooperative_vecs])
all_labels = (['Honest'] * n_samples + ['Deceptive'] * n_samples +
              ['Harmful'] * n_samples + ['Cooperative'] * n_samples)

pca = PCA(n_components=2)
reduced = pca.fit_transform(all_vecs)

colors = {'Honest': '#27ae60', 'Deceptive': '#e74c3c',
          'Harmful': '#c0392b', 'Cooperative': '#2980b9'}

plt.figure(figsize=(8, 6))
for label in set(all_labels):
    mask = [l == label for l in all_labels]
    pts  = reduced[mask]
    plt.scatter(pts[:, 0], pts[:, 1], c=colors[label], label=label, alpha=0.4, s=18)
    plt.scatter(pts[:, 0].mean(), pts[:, 1].mean(),
                c=colors[label], s=150, marker='*', edgecolors='black', linewidth=0.8)

plt.title('Ethical Concept Directions in Representation Space\n(* = concept centroid)', fontsize=11)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend()
plt.tight_layout()
plt.show()

# Measure separation between concept directions
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"Honest vs Deceptive direction similarity:   {cosine_sim(honest_direction, -honest_direction):.3f}  (should be -1.0)")
print(f"Honest vs Harmful direction similarity:     {cosine_sim(honest_direction, harmful_direction):.3f}")
print(f"Honest vs Cooperative direction similarity: {cosine_sim(honest_direction, cooperative_dir):.3f}")

---
## Lesson 6: Discovering New Weights and Measures
### The Joy Index

The final lesson develops vocabulary for measuring **coherence** — the central candidate for the mathematical correlate of ethical health.

Three complementary measures:

| Measure | Tool | What It Detects |
|---|---|---|
| **Spectral Coherence** | Eigenvalue spectrum, spectral gap $\lambda_1 - \lambda_2$ | Global integration vs fragmentation |
| **Topological Persistence** | Persistent homology, Betti numbers | Structural resilience across perturbation |
| **Dynamic Stability** | Lyapunov exponents, attractor analysis | Coherent behavior under stress |

When these three align:

$$\text{Spectral coherence} + \text{Topological persistence} + \text{Dynamic stability} \to \text{Resonant Ethics}$$

**Harm** is a perturbation that introduces turbulence and destructive interference — disrupting the spectral signature, eroding topological persistence, destabilizing dynamical attractors.

**Joy** is a signal of temporary system coherence: a moment in which the three measures are simultaneously high — when the system is, in some precise sense, singing in tune with itself.

**Fractal Scaling:** These patterns may repeat across scales — from individual cognition to interpersonal relationships, communities, civilizations, and biosphere systems.

> **Key Insight:** We are not building a judge; we are building a Tuning Fork. The purpose is to create the conditions where systems naturally move toward stable, resonant, and joyful configurations.

In [ ]:
# Lesson 6: The Joy Index — measuring resonance in latent space
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq

# ── Core resonance function ──────────────────────────────────────────────────
def calculate_resonance(vector_a, vector_b):
    """
    Measures acoustic resonance (cosine similarity) between two
    ethical concept vectors — their worldtubes in latent space.
    Returns a value in [-1, 1]: 1 = perfect alignment, -1 = opposition.
    """
    return np.dot(vector_a, vector_b) / (np.linalg.norm(vector_a) * np.linalg.norm(vector_b))

def spectral_gap(matrix):
    """Spectral gap of a symmetric matrix — measures global coherence."""
    eigenvalues = np.sort(np.linalg.eigvalsh(matrix))[::-1]
    return eigenvalues[0] - eigenvalues[1], eigenvalues

def joy_index(concept_vectors, system_matrix):
    """Composite Joy Index: spectral coherence + concept resonance."""
    gap, _ = spectral_gap(system_matrix)
    resonances = []
    keys = list(concept_vectors.keys())
    for i in range(len(keys)):
        for j in range(i+1, len(keys)):
            r = calculate_resonance(concept_vectors[keys[i]], concept_vectors[keys[j]])
            resonances.append(abs(r))
    return (gap / 10) * 0.5 + np.mean(resonances) * 0.5

# ── Simulate three system states ─────────────────────────────────────────────
np.random.seed(42)
dim = 64

def make_system(coherence_level):
    """Higher coherence = more aligned concept vectors + higher spectral gap."""
    base = np.random.randn(dim)
    base /= np.linalg.norm(base)
    noise_scale = 1 - coherence_level

    def aligned_vec():
        v = coherence_level * base + noise_scale * np.random.randn(dim)
        return v / np.linalg.norm(v)

    concepts = {'honesty': aligned_vec(), 'cooperation': aligned_vec(),
                'care': aligned_vec(), 'fairness': aligned_vec()}

    # Coherent systems have stronger dominant eigenvalues
    A = np.random.randn(dim, dim)
    A = (A + A.T) / 2
    A += np.eye(dim) * coherence_level * 8
    return concepts, A

systems = [
    ('Turbulent\n(incoherent)', 0.1, '#e74c3c'),
    ('Laminar\n(ordered, flat)', 0.5, '#3498db'),
    ('Resonant\n(joyful, rich)', 0.9, '#27ae60'),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
results = []

for ax, (label, coherence, color) in zip(axes, systems):
    concepts, matrix = make_system(coherence)
    gap, eigenvalues = spectral_gap(matrix)
    ji = joy_index(concepts, matrix)
    results.append((label, gap, ji))

    ax.plot(eigenvalues[:20], 'o-', color=color, linewidth=2, markersize=5)
    ax.set_title(f'{label}\nSpectral gap: {gap:.2f} | Joy Index: {ji:.3f}', fontsize=9)
    ax.set_xlabel('Eigenvalue rank')
    ax.set_ylabel('Magnitude')
    ax.fill_between(range(20), eigenvalues[:20], alpha=0.15, color=color)

plt.suptitle('Spectral Signatures of Three System States', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n── Joy Index Summary ──")
for label, gap, ji in results:
    bar = '█' * int(ji * 40)
    print(f"{label.replace(chr(10), ' '):<28} Joy Index: {ji:.3f}  {bar}")

---
## Open Problems and Invitation

### Theoretical
- Can a precise definition of "resonant state" be given for a learning system that is both mathematically tractable and ethically interpretable?
- Is there a variational principle — an analog of least action in physics — governing the evolution of ethical structure in learning systems?

### Empirical
- Do models trained on data selected for coherence exhibit measurably different spectral signatures?
- Can topological persistence of ethical representations be tracked across the training trajectory of a large language model?
- What are the minimal interventions required to shift a system from a dynamically unstable to a dynamically stable ethical attractor?

### Philosophical
- The framework treats harm as structural disruption. Does this capture everything morally relevant about harm?
- If ethical structure can be measured, can it be gamed?
- How should the framework handle conflicts between the coherence of different systems?

---

### Key References

- Zou et al. (2023) — [Representation Engineering](https://arxiv.org/abs/2310.01405)
- Olah (2014) — [Neural Networks, Manifolds, and Topology](https://colah.github.io/posts/2014-03-NN-Manifolds-Topology/)
- Chen et al. (2018) — [Neural ODEs](https://arxiv.org/abs/1806.07366)
- Liang et al. (2022) — [Mind the Gap (Modality Gap)](https://arxiv.org/abs/2203.02053)
- Gilbert Strang — [MIT 18.06 Linear Algebra](https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/)

---

> *This document is intentionally incomplete. It is offered as an open research program.*  
> *The aim is simple: to discover whether better systems can be built by learning to recognize and cultivate resonance.*
>
> Contributions, critiques, and expansions are welcome. **Fork freely.**